In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import pandas as pd
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split

In [ ]:
!ls

In [ ]:
df = pd.read_csv("training_data.csv")
df.head()

In [ ]:
df["priority"].value_counts()

In [ ]:
label_map = {
    "low": 0,
    "medium": 1,
    "high": 2
}
df["label"] = df["priority"].map(label_map)

In [ ]:
df["label"].value_counts()

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["description"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
train_encodings = tokenizer(
    train_texts, truncation=True, padding=True
)
val_encodings = tokenizer(
    val_texts, truncation=True, padding=True
)

In [ ]:
class PriorityDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = PriorityDataset(train_encodings, train_labels)
val_dataset = PriorityDataset(val_encodings, val_labels)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
model.save_pretrained("priority_ai_model")
tokenizer.save_pretrained("priority_ai_model")

In [ ]:
# IMPROVED: Pure AI-based Priority Classification System
# This replaces the previous rule-based approach with a clean AI-only solution
# The trained DistilBERT model handles all priority decisions based on learned patterns

# Pure AI-based priority prediction function
def predict_priority(text):
    """Predict priority using the trained DistilBERT model"""
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=512
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(predictions, dim=-1).item()
    
    # Convert numerical prediction back to priority labels
    label_names = {0: "low", 1: "medium", 2: "high"}
    return label_names[predicted_class]

def final_priority_decision(data):
    """Main function to determine priority using AI model"""
    age = data.get("age", 0)
    gender = data.get("gender", "Unknown")
    income = data.get("income", "Unknown")
    disability = data.get("disability", "No")
    dependents = data.get("dependents", 0)
    description = data.get("description", "")
    
    # Format data as text for the AI model (same format as training data)
    text = (
        f"Age: {age} Gender: {gender} Income: {income} "
        f"Disability: {disability} Dependents: {dependents} "
        f"Description: {description}"
    )
    
    # Use AI model for prediction
    priority = predict_priority(text)
    return priority.capitalize()  # Return "High", "Medium", or "Low"

In [ ]:
# Test Case 1: Elderly with disability
test1 = {
    "age": 80,
    "income": "None",
    "disability": "Yes",
    "dependents": 0,
    "description": "Elderly person with mobility issues"
}

print(f"Test 1 Result: {final_priority_decision(test1)}")

In [ ]:
# Test Case 2: Young person with dependents
test2 = {
    "age": 25,
    "gender": "Female",
    "income": "Low",
    "disability": "No",
    "dependents": 3,
    "description": "Single mother with three children"
}

print(f"Test 2 Result: {final_priority_decision(test2)}")

In [ ]:
# Test Case 3: Direct AI prediction
test_text = "Age: 45 Gender: Male Income: Medium Disability: No Dependents: 1 Description: Working father with stable job"
print(f"Direct AI prediction: {predict_priority(test_text)}")

In [ ]:
# Save the model and create backup
!zip -r priority_ai_model_backup.zip priority_ai_model results training_data.csv